In [ ]:
# way 1
# %%capture
# # Installs Unsloth, Xformers (Flash Attention) and all other packages!
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# # way 2
# !wget -q https://github.com/conda-forge/miniforge/releases/download/25.3.0-3/Miniforge3-25.3.0-3-Linux-x86_64.sh

# # Установите Miniforge
# !bash Miniforge3-25.3.0-3-Linux-x86_64.sh -b -f -p /usr/local/miniforge

# # Добавьте в PATH
# import os
# os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]

# # Инициализация
# !source /usr/local/miniforge/bin/activate

In [ ]:
# way 3
# !conda create --name unsloth_env \
#     python=3.11 \
#     pytorch-cuda=12.1 \
#     pytorch cudatoolkit xformers -c pytorch -c nvidia -c xformers \
#     -y
# !conda activate unsloth_env

# !pip install unsloth

In [ ]:
import unsloth
from unsloth import FastModel
import torch

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3n-E4B-it",
    dtype = None, # None for auto detection
    max_seq_length = 1024, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

In [ ]:
from transformers import TextStreamer
# Helper function for inference
def do_gemma_3n_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

In [ ]:
audio_file = "./chunk_2.wav"

messages = [{
    "role" : "user",
    "content": [
        { "type": "audio", "audio" : audio_file },
        { "type": "text",  "text" : "What is this audio about?" }
    ]
}]

In [ ]:
do_gemma_3n_inference(messages, max_new_tokens = 256)